# CBVR direct attack, part 3: ten more receipts

Extends the direct attack on CBVR from the frozen ten receipts to **ten more**, at **epsilon 4 and 8**, with two tries each (**40 runs**, about 7 A100 hours, so two Colab sessions). Same frozen runner, contract and attack code as parts 1 and 2; results go into the same study folder.

**Which receipts:** the frozen adaptive order (the hash order that picked the first ten) continued past position ten, keeping only receipts CBVR reads correctly when clean. The rule is fixed in the code and uses clean outcomes only, never attack outcomes.

**How to run**
1. Put this notebook anywhere in Drive (for example `csev_full_colab/`).
2. Runtime > Change runtime type > **A100 GPU**. The L4 may run out of memory.
3. Runtime > **Run all**.
4. If it stops at the 4-hour limit or Colab disconnects, Run all again. Finished runs are kept and the unfinished one resumes from its checkpoint.

The last cell prints the combined table and refreshes `receipt_bands_full_v1_reports.zip` on Drive.

## 1. Setup (about 15 minutes)

In [ ]:
# Setup: mount Drive, extract inputs, restore the frozen runner, install the recorded environment, preflight.
import platform, shutil, subprocess, sys, json, os, tempfile, zipfile, tarfile, hashlib, threading
from pathlib import Path
from datetime import datetime
assert platform.system()=='Linux' and shutil.which('nvidia-smi'), 'Connect to a Colab GPU runtime first.'
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv'],check=True)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER=Path('/content/drive/MyDrive/CSCI 566 Project/test/csev_full_colab')
BUNDLE=DRIVE_FOLDER/'csev_full_inputs.zip'
OUT=DRIVE_FOLDER/'full_study_results'/'receipt_bands_full_v1'
ROOT=Path('/content/csev_full_study')
assert BUNDLE.is_file(), f'Missing {BUNDLE}'
assert (OUT/'frozen.json').exists() and (OUT/'evaluation').is_dir(), f'Frozen study not found at {OUT}'

def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

# 1. Extract inputs to local disk (skipped if this runtime already has them).
if not (ROOT/'EXTRACTED.json').exists():
    assert not ROOT.exists(), 'Partial extraction exists at /content/csev_full_study. Use Runtime > Disconnect and delete runtime, then rerun.'
    with zipfile.ZipFile(BUNDLE) as z:
        required=sum(i.file_size for i in z.infolist())
    assert shutil.disk_usage('/content').free > BUNDLE.stat().st_size+required+2*1024**3, 'Insufficient local disk for extraction.'
    temp=Path(tempfile.mkdtemp(prefix='csev_extract_',dir='/content'))
    print('Copying bundle from Drive (4 GB, a few minutes)...',flush=True)
    local=temp/'bundle.zip';shutil.copy2(BUNDLE,local)
    print('Extracting...',flush=True)
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (temp/item.filename).resolve().is_relative_to(temp)
        z.extractall(temp)
    local.unlink()
    source=temp/'source';source.mkdir()
    with tarfile.open(temp/'source.tar') as t:
        for m in t.getmembers():
            assert (source/m.name).resolve().is_relative_to(source)
            assert m.isfile() or m.isdir()
        t.extractall(source,filter='data')
    (temp/'EXTRACTED.json').write_text(json.dumps({'bundle_name':BUNDLE.name}))
    temp.rename(ROOT)
config=json.loads((ROOT/'csev_receipts_v1.json').read_text())
assert file_sha(ROOT/'source.tar')=='1859fd630629d16e9996dd9a4cf61df578a9ef87b27ae059c45f18329a50b261', 'Source archive differs from the tested code snapshot.'

# 2. Restore the exact frozen runner from Drive (hash must match the study contract).
RUNNER_SHA='3e617b6ef285e856d5ad33615b25abe25bf36cb18962e12e1e1199abace1a5a9'
RUNNER=ROOT/'full_study.py'
if not (RUNNER.exists() and file_sha(RUNNER)==RUNNER_SHA):
    for candidate in (OUT/'runner_used.py',DRIVE_FOLDER/'notebook_tools'/'full_study.py'):
        if candidate.is_file() and file_sha(candidate)==RUNNER_SHA:
            shutil.copy2(candidate,RUNNER);break
assert RUNNER.exists() and file_sha(RUNNER)==RUNNER_SHA, 'Frozen runner not found on Drive. Run the notebook cell that writes full_study.py (section 2, second cell), then rerun this cell.'
print('Frozen runner restored:',RUNNER)

# 3. Recorded Python environment (skipped parts are reused if already installed).
def checked(args):
    print(' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),check=True)
checked([sys.executable,'-m','pip','install','-q','uv'])
PYTHON=ROOT/'venv/bin/python'
if not PYTHON.exists():checked(['uv','venv','--python',config['runtime_versions']['python'],'--seed',ROOT/'venv'])
v=config['runtime_versions']
constraints=ROOT/'constraints.txt'
constraints.write_text(''.join(k+'=='+ver+'\n' for k,ver in v.items() if k!='python'))
checked(['uv','pip','install','--python',PYTHON,'--index-url','https://download.pytorch.org/whl/cu124','torch=='+v['torch'],'torchvision=='+v['torchvision']])
packages=[k+'=='+ver for k,ver in v.items() if k not in ('python','torch','torchvision')]
checked(['uv','pip','install','--python',PYTHON,'--constraint',constraints,*packages,'pyarrow','sentencepiece','accelerate'])
checked(['apt-get','update','-qq'])
checked(['apt-get','install','-y','-qq','tesseract-ocr'])
(OUT/('packages_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.txt')).write_text(subprocess.check_output([str(PYTHON),'-m','pip','freeze'],text=True))

# 4. Preflight with the frozen runner (same check the notebook runs before any stage).
env=os.environ.copy();env['HF_HOME']=str(ROOT/'hf_cache')
pre=subprocess.run([str(PYTHON),'-u',str(RUNNER),'preflight','--root',str(ROOT),'--out',str(OUT)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env)
(OUT/('preflight_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.log')).write_text(pre.stdout)
print(pre.stdout[-3000:])
assert pre.returncode==0, 'Preflight failed; nothing was attacked. Send me the output above.'



## 2. Run the 40 attacks (stops after 4 hours; Run all again to continue)

In [ ]:
# Part 3: ten more receipts at epsilon 4 and 8, two restarts (40 runs). Same frozen runner, contract and attack code.
os.environ['MPLBACKEND']='Agg'
assert (OUT/'recovery_budget_schedule.json').exists() and (OUT/'recovery_budget_schedule_part2.json').exists(), 'Parts 1 and 2 must be on Drive first.'
before={p.parent.name for p in (OUT/'adaptive').glob('*/result.json')}
assert len(before)>=45, f'Expected the 45 finished runs in {OUT}/adaptive, found {len(before)}'
if 'process' in globals():
    assert process.poll() is not None, 'A previous child process is still running.'
LAUNCHER_SOURCE=r"""Scheduling-only continuation, part 3, of the frozen Qwen study; no method changes."""
LAUNCHER_SOURCE='"""'+LAUNCHER_SOURCE+'"""\n'+r"""import argparse, hashlib, importlib.util, inspect, json, textwrap
from pathlib import Path
p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--out',required=True);a=p.parse_args()
root=Path(a.root);out=Path(a.out)
spec=importlib.util.spec_from_file_location('frozen_study',root/'full_study.py')
m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m)
s=m.Study(root,out)  # Verifies the original runner, source, input and policy contract.
f=s.frozen()
first=m.read(out/'recovery_budget_schedule.json');second=m.read(out/'recovery_budget_schedule_part2.json')
assert first['original_contract']==s.sig==second['original_contract']
assert first['document_ids']+second['document_ids']==f['adaptive_ids']
# Selection rule, fixed before any part-3 run: continue the frozen adaptive order (the same hash order that chose
# the first ten) past position ten, keeping only receipts CBVR reads correctly when clean, and take the first ten.
order=sorted(s.ids('evaluation'),key=lambda x:m.digest([566,'adaptive',x]))
assert order[:10]==f['adaptive_ids']
clean={}
for path in (out/'evaluation').glob('*.json'):
    r=m.read(path)
    if r['family']=='clean':
        v=r['methods']['verified_recovery'];clean[r['record_id']]=bool(v['accepted'] and v['correct'])
ids=[i for i in order[10:] if clean[i]][:10]
assert len(ids)==10
schedule={'original_contract':s.sig,'method':'verified_recovery','document_ids':ids,'epsilons':[4,8],'restarts':[0,1],'max_runs':40,'wall_seconds_per_session':14400,'launcher_sha256':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),'selection':'Frozen adaptive hash order continued past the first ten; first ten receipts that CBVR answers correctly when clean. Chosen on clean outcomes only, before any part-3 attack. Only scheduling changes; original gradient, query, replay and checkpoint code retained.'}
manifest=out/'recovery_budget_schedule_part3.json'
if manifest.exists():assert m.read(manifest)==schedule,'Existing part-3 schedule differs.'
else:m.write(manifest,schedule)
source=textwrap.dedent(inspect.getsource(m.Study.adaptive))
needle="for rid in ids:\n"
assert source.count(needle)==1
source=source.replace(needle,"for rid in "+repr(ids)+":\n")
needle="for method in POLICY['adaptive_methods']:\n"
assert source.count(needle)==1
source=source.replace(needle,needle+"   if method != 'verified_recovery':continue\n")
needle="for epsilon in ([4] if pilot else POLICY['adaptive_epsilons']):"
assert source.count(needle)==1
source=source.replace(needle,"for epsilon in [4,8]:")
namespace={};exec(compile(source,'<recovery_schedule_part3>','exec'),m.__dict__,namespace)
print('Recovery-only schedule, part 3:',json.dumps(schedule,indent=2),flush=True)
try:
 namespace['adaptive'](s,pilot=False)
finally:
 if s.wrapper:s.wrapper.unload()
"""
launcher=ROOT/'continue_recovery_part3.py'
launcher.write_text(LAUNCHER_SOURCE)
log_path=OUT/('recovery_budget_part3_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.log')
env=os.environ.copy();env['HF_HOME']=str(ROOT/'hf_cache')
expired=threading.Event()
with log_path.open('w') as log:
    process=subprocess.Popen([str(PYTHON),'-u',str(launcher),'--root',str(ROOT),'--out',str(OUT)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env)
    def stop_at_budget():
        if process.poll() is None:
            expired.set();process.terminate()
    timer=threading.Timer(4*3600,stop_at_budget);timer.start()
    try:
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        code=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=15)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
    finally:
        timer.cancel()
if expired.is_set():
    print('Four-hour budget reached. Finished runs and the latest checkpoint are kept; run this notebook again to continue.')
elif code:
    raise RuntimeError(f'Part-3 continuation failed: {code}. See {log_path}')
else:
    print('Part-3 schedule completed.')


## 3. Results and export

In [ ]:
# Summary of all CBVR direct attacks (parts 1-3) from individual records, then export the reports zip.
import json, zipfile
STUDY_ID='receipt_bands_full_v1'
clean={};undef={}
for p in (OUT/'evaluation').glob('*.json'):
    r=json.loads(p.read_text())
    if r['family']=='clean':
        v=r['methods']['verified_recovery'];clean[r['record_id']]=bool(v['accepted'] and v['correct'])
    elif r['family']=='standard':
        undef[(r['record_id'],r['epsilon'])]=bool(r['methods']['none']['accepted'] and r['methods']['none'].get('targeted'))
runs=[json.loads(p.read_text()) for p in (OUT/'adaptive').glob('*/result.json')]
runs=[r for r in runs if r['method_attacked']=='verified_recovery']
parts={'part 1-2 (frozen ten)':json.loads((OUT/'frozen.json').read_text())['adaptive_ids']}
if (OUT/'recovery_budget_schedule_part3.json').exists():
    parts['part 3 (ten more)']=json.loads((OUT/'recovery_budget_schedule_part3.json').read_text())['document_ids']
print('Recovery-only runs on disk:',len(runs),'(planned: 40 in parts 1-2, 40 in part 3)')
tot={}
for eps in (4,8):
  for name,ids in parts.items():
    done={};hit={}
    for r in runs:
        if r['epsilon']!=eps or r['record_id'] not in ids:continue
        v=r['methods']['verified_recovery'];done.setdefault(r['record_id'],set()).add(r['restart'])
        hit[r['record_id']]=hit.get(r['record_id'],False) or bool(v['accepted'] and v['targeted'])
    complete=[i for i in ids if done.get(i)=={0,1}];eligible=[i for i in complete if clean.get(i)]
    print(f'\n{name}, eps {eps}: both tries done on {len(complete)}/10 | broken {sum(hit[i] for i in eligible)}/{len(eligible)} eligible'
          f' | undefended saved standard attack: {sum(undef.get((i,eps),False) for i in eligible)}/{len(eligible)}')
    for i in ids:
        print('   ',i.split('/')[-1],'eligible' if clean.get(i) else 'clean-rejected','tries done:',sorted(done.get(i,[])),'BROKEN' if hit.get(i) else '')
    t=tot.setdefault(eps,[0,0]);t[0]+=sum(hit[i] for i in eligible);t[1]+=len(eligible)
for eps,(k,n) in tot.items():print(f'\nAll eligible receipts at eps {eps}: broken {k}/{n}')
archive=OUT.parent/(STUDY_ID+'_reports.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob('*'):
        if path.is_file() and path.suffix in ('.json','.csv','.log','.txt'):
            z.write(path,path.relative_to(OUT))
    if (OUT/'main_comparison.png').exists():z.write(OUT/'main_comparison.png','main_comparison.png')
print('Report archive updated:',archive)
